# **3. Baseline Model**

In [40]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable
import sklearn
import feature_engine
import scipy
from scipy import stats
from pathlib import Path
import pickle

%matplotlib inline
sns.set_style('darkgrid')


In [41]:
# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [42]:
# define file path for processed data
parent_path = Path.cwd().parent
file_path = parent_path.joinpath("models", "processed.pkl")

In [43]:
# Load the processed data from the previous step
print(f"Loading processed data...")
with open(file_path, "rb") as f:
    processed_data = pickle.load(f)

Loading processed data...


In [44]:
processed_data.keys()

dict_keys(['X_train', 'y_train', 'X_test', 'test_id', 'numeric_features', 'ordinal_features', 'categorical_features', 'boolean_features', 'year_features'])

In [45]:
# Extract features and target variable
X_train = processed_data["X_train"]
y_train = processed_data["y_train"]
X_test = processed_data["X_test"]
numeric_feat = processed_data["numeric_features"]
ordinal_feat = processed_data["ordinal_features"]
categorical_feat = processed_data["categorical_features"]
bool_feat = processed_data["boolean_features"]
year_feat = processed_data["year_features"]

In [46]:
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

Training data shape: (1459, 83)
Test data shape: (1457, 83)


#### **1. Feature Selection**

In [88]:
ordinal_feat_remove = ['TotRmsAbvGrd', 'GarageCars', 'FullBath']
numeric_feat_remove = ['1stFlrSF']
year_feat_remove = ['GarageYrBlt']
cat_feat_remove = ['SaleCondition','MSZoning','PavedDrive','LotShape','SaleType','HouseStyle','Exterior1st',
'RoofStyle','Exterior2nd','BldgType','LandContour','ExterCond','LotConfig',
'RoofMatl','Condition1','Heating','Functional','Condition2','LandSlope', 'Utilities', 'Street', 'Alley', 
'MasVnrType', 'Electrical', 'GarageType', 'MiscFeature']



In [100]:
X_train[cat_var].describe()

,Neighborhood,Foundation,CentralAir
count,1459,1459,1459
unique,25,6,2
top,NAmes,PConc,Y
freq,225,646,1364


In [89]:
# Remove features which heplp constructing the new features in the previous step
ordinal_var = [feat for feat in ordinal_feat if feat not in ordinal_feat_remove]
numeric_var = [feat for feat in numeric_feat if feat not in numeric_feat_remove]
year_var = [feat for feat in year_feat if feat not in year_feat_remove]
cat_var = [feat for feat in categorical_feat if feat not in cat_feat_remove]

In [92]:
# Preference 1: Correlation Analysis: compute correlation between numeric features and the target variable
correlations = X_train[ordinal_var + numeric_var + year_var].corrwith(y_train).abs().sort_values(ascending=False)
print("Top 20 correlations of features with the target variable:")
print(correlations[:20])

Top 20 correlations of features with the target variable:
OverallQualSquared         0.820467
LivAreaQual                0.795007
OverallQual                0.793055
TotalSF                    0.765099
NeighborhoodMedianPrice    0.720926
GrLivArea                  0.698047
ExterQual                  0.684381
KitchenQual                0.660666
TotalBath                  0.633484
GarageArea                 0.624160
BsmtQual                   0.585748
HouseAge                   0.573607
QualCond                   0.565831
GarageFinish               0.549590
YearBuilt                  0.523110
FireplaceQu                0.520656
RemodeledAge               0.513813
YearRemodAdd               0.507283
BathToBedRatio             0.483330
Fireplaces                 0.466967
dtype: float64


In [97]:
# Prefrence 2: Use Ridge Regression to compute feature importance
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Transformes to cast boolean to int
bool_to_int = FunctionTransformer(lambda x: x.astype(int))

# Define preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), numeric_var),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['Neighborhood']),
        ("bool", bool_to_int, bool_feat),
        ("ordinal", "passthrough", ordinal_var),
        ("year", "passthrough", year_var)
    ]
)

# build pipeline
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=10.0))
])
ridge_pipeline.fit(X_train, y_train)

# Extract features names after one-hot encoding
features_name = (
    numeric_var +
    list(ridge_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(['Neighborhood'])) +
    bool_feat + ordinal_var + year_var
) 

# Extract coefficients from the Ridge model
ridge_coef = ridge_pipeline.named_steps['ridge'].coef_

feature_importance = pd.DataFrame({
    "Feature": features_name,
    "Importance": ridge_coef
})
feature_importance.sort_values(by="Importance", ascending=False, inplace=True, ignore_index=True)
print("Top 20 features by importance from Ridge Regression:")
print(feature_importance.head(20))


Top 20 features by importance from Ridge Regression:
                    Feature    Importance
0      Neighborhood_NoRidge  23192.649099
1                 GrLivArea  20346.015685
2      Neighborhood_StoneBr  16325.685608
3                   TotalSF  14859.371193
4                 TotalBath  12356.947167
5       Neighborhood_BrDale  12249.682038
6                BsmtFinSF1  11625.200742
7               OverallCond  10417.820609
8      Neighborhood_BrkSide  10378.051101
9   NeighborhoodMedianPrice   9760.863003
10     Neighborhood_Crawfor   9632.721739
11     Neighborhood_MeadowV   8683.177624
12                  LotArea   8322.862547
13              KitchenQual   6336.642472
14              OpenPorchSF   6309.252460
15               Fireplaces   5814.405951
16                 BsmtQual   5804.562820
17             BsmtExposure   4825.537302
18       OverallQualSquared   4733.433232
19               GarageArea   4691.153666
